In [2]:
import tensorflow as tf
import pandas as pd
# from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report
import joblib

I0000 00:00:1795285947.318248   45454 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
import pandas as pd

train_df = pd.read_csv("../data/amazon_polarity_train.csv", nrows=8000)
test_df = pd.read_csv("../data/amazon_polarity_test.csv", nrows=200)

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

Train shape: (8000, 3)
Test shape:  (200, 3)


In [4]:
train_texts = [
    f"{str(t)} {str(c)}".strip()
    for t, c in zip(train_df.get("title", [""] * len(train_df)), train_df.get("content", [""] * len(train_df)))
]
test_texts = [
    f"{str(t)} {str(c)}".strip()
    for t, c in zip(test_df.get("title", [""] * len(test_df)), test_df.get("content", [""] * len(test_df)))
]

y_train = train_df["label"].values
y_test = test_df["label"].values

In [5]:
model_ckpt = "distilbert-base-uncased"
print(f"Loading feature-extraction pipeline with '{model_ckpt}'...")

extractor = pipeline(
    task="feature-extraction",
    model=model_ckpt,
    tokenizer=model_ckpt,
    truncation=True,
    max_length=128,
)
print("Feature-extraction pipeline ready.")

Loading feature-extraction pipeline with 'distilbert-base-uncased'...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2917.07it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Feature-extraction pipeline ready.


In [7]:
import numpy as np
from sklearn.linear_model import LogisticRegression

In [8]:
def extract_cls_embeddings(texts, batch_size=32):
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        outputs = extractor(batch)
        # outputs[i] is [seq_len, hidden_size]; index [0][0] takes the [CLS] token vector
        batch_vecs = [np.asarray(seq[0][0], dtype=np.float32) for seq in outputs]
        all_embeddings.extend(batch_vecs)
    return np.vstack(all_embeddings)

print("Extracting embeddings for training reviews...")
X_train_bert = extract_cls_embeddings(train_texts, batch_size=32)

print("Extracting embeddings for test reviews...")
X_test_bert = extract_cls_embeddings(test_texts, batch_size=32)

print(f"X_train_bert shape: {X_train_bert.shape} | X_test_bert shape: {X_test_bert.shape}")

clf_bert = LogisticRegression(max_iter=1000, random_state=42)
clf_bert.fit(X_train_bert, y_train)
print("BERT + Logistic Regression sentiment classifier trained.")

Extracting embeddings for training reviews...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Extracting embeddings for test reviews...
X_train_bert shape: (8000, 768) | X_test_bert shape: (200, 768)
BERT + Logistic Regression sentiment classifier trained.


In [9]:
ticket_samples = [
    ("My package never arrived and tracking hasn't updated", "Shipping"),
    ("Where is my order? It is five days past delivery estimate", "Shipping"),
    ("Item arrived cracked and scratched inside the box", "Returns & Refunds"),
    ("I want to return this item and get a full refund", "Returns & Refunds"),
    ("My card was charged twice for the same order", "Billing"),
    ("The invoice shows an incorrect tax charge", "Billing"),
    ("The app crashes every time I try to check out", "Technical Support"),
    ("I cannot reset my password or log into my account", "Technical Support"),
]

ticket_texts = [text for text, _ in ticket_samples]
ticket_labels = [label for _, label in ticket_samples]

X_tickets = extract_cls_embeddings(ticket_texts, batch_size=8)
ticket_router = LogisticRegression(max_iter=1000, random_state=42)
ticket_router.fit(X_tickets, ticket_labels)

sample_query = ["The blender stopped turning on after a week, I want a refund"]
routed_label = ticket_router.predict(extract_cls_embeddings(sample_query, batch_size=1))[0]
print(f"Sample ticket: '{sample_query[0]}'")
print(f"Predicted route: {routed_label}")

Sample ticket: 'The blender stopped turning on after a week, I want a refund'
Predicted route: Returns & Refunds


In [12]:
y_pred_bert = clf_bert.predict(X_test_bert)
bert_acc = accuracy_score(y_test, y_pred_bert)

print(f"BERT Pipeline Accuracy (200 test samples): {bert_acc:.4f}\n")
print("Per-Class Evaluation Report:")
print(classification_report(y_test, y_pred_bert, target_names=["Negative (0)", "Positive (1)"]))

tfidf_vec_path = "../models/tfidf_vectorizer.pkl"
tfidf_clf_path = "../models/sentiment_logistic_regression.pkl"

import os

if os.path.exists(tfidf_vec_path) and os.path.exists(tfidf_clf_path):
    tfidf_vec = joblib.load(tfidf_vec_path)
    tfidf_clf = joblib.load(tfidf_clf_path)
    X_test_tfidf = tfidf_vec.transform(test_texts)
    tfidf_acc = accuracy_score(y_test, tfidf_clf.predict(X_test_tfidf))
    print(f"Saved TF-IDF Baseline Accuracy: {tfidf_acc:.4f}")
    print(f"Accuracy Lift (BERT - TF-IDF):  {bert_acc - tfidf_acc:+.4f}")
else:
    print("No saved TF-IDF baseline found at 'models/'; skipping comparison.")

BERT Pipeline Accuracy (200 test samples): 0.8800

Per-Class Evaluation Report:
              precision    recall  f1-score   support

Negative (0)       0.85      0.89      0.87        91
Positive (1)       0.90      0.87      0.89       109

    accuracy                           0.88       200
   macro avg       0.88      0.88      0.88       200
weighted avg       0.88      0.88      0.88       200

Saved TF-IDF Baseline Accuracy: 0.8250
Accuracy Lift (BERT - TF-IDF):  +0.0550


In [14]:
joblib.dump(clf_bert, "../models/bert_sentiment_clf.pkl")
# joblib.dump(ticket_router, "../models/bert_pipeline/bert_ticket_router.pkl")

print("Saved '../modelsbert_sentiment_clf.pkl'")
# print("Saved 'models/bert_pipeline/bert_ticket_router.pkl'")

Saved '../modelsbert_sentiment_clf.pkl'
